# Dataset : CIFAR-10  
            60,000 colour images (32×32×3) across 10 object classes:
            airplane, automobile, bird, cat, deer,
            dog, frog, horse, ship, truck
  Libraries: NumPy · Pandas · Matplotlib · Seaborn · Scikit-Learn · TensorFlow

In [ ]:
# Imports
import os, time, warnings
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches

import seaborn as sns
warnings.filterwarnings("ignore")


In [ ]:
from sklearn.linear_model    import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier
from sklearn.svm             import SVC
from sklearn.decomposition   import PCA
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import (
    accuracy_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)
from sklearn.pipeline        import Pipeline

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, Model
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, LearningRateScheduler
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# Output directory
OUT = "cifar10_outputs"
os.makedirs(OUT, exist_ok=True)

CLASS_NAMES = ["airplane","automobile","bird","cat","deer",
               "dog","frog","horse","ship","truck"]

In [ ]:
# Colour palette
BG_DARK  = "#0d0d1a"
BG_PANEL = "#16213e"
BG_MID   = "#1a1a2e"
ACCENT   = "#4C72B0"
GREEN    = "#2ecc71"
RED      = "#e74c3c"

def _style(fig, *axes):
    fig.patch.set_facecolor(BG_DARK)
    for ax in axes:
        ax.set_facecolor(BG_PANEL)
        ax.tick_params(colors="white")
        for sp in ax.spines.values():
            sp.set_edgecolor("#444466")

print("="*72)
print("  ML JOURNEY — CIFAR-10 Image Classification")
print(f"  TensorFlow {tf.__version__}  |  NumPy {np.__version__}  |  Pandas {pd.__version__}")
print("="*72)

results = {}   # name → {accuracy, train_time, history, type}




In [ ]:
#  DATA LOADING & EXPLORATORY DATA ANALYSIS
(X_tr_raw, y_tr_raw), (X_te_raw, y_te_raw) = keras.datasets.cifar10.load_data()
y_tr_raw = y_tr_raw.flatten()
y_te_raw = y_te_raw.flatten()

print(f"  Train : {X_tr_raw.shape}  |  Test : {X_te_raw.shape}")
print(f"  Pixel range : [{X_tr_raw.min()}, {X_tr_raw.max()}]")

In [ ]:
# Pandas EDA
flat = X_tr_raw.reshape(50_000, -1)          # 50000 × 3072
df_eda = pd.DataFrame({
    "label"   : y_tr_raw,
    "class"   : [CLASS_NAMES[i] for i in y_tr_raw],
    "mean_R"  : flat[:, 0:1024].mean(1),
    "mean_G"  : flat[:, 1024:2048].mean(1),
    "mean_B"  : flat[:, 2048:].mean(1),
    "brightness": flat.mean(1),
})
print("\n  Pandas head (first 5 rows):")
print(df_eda.head())
print("\n  Class distribution (training):")
print(df_eda["class"].value_counts().to_string())
print("\n  Channel statistics:")
print(df_eda[["mean_R","mean_G","mean_B","brightness"]].describe().round(2))


In [ ]:
# sample grid + class distribution
fig = plt.figure(figsize=(22, 11))
_style(fig)
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# Sample grid (3 × 10)
ax_grid = fig.add_subplot(gs[0, :2])
ax_grid.axis("off")
ax_grid.set_facecolor(BG_DARK)
ax_grid.set_title("Sample Images — 3 per Class", color="white", fontsize=14, pad=8)
for ci, cname in enumerate(CLASS_NAMES):
    idxs = np.where(y_tr_raw == ci)[0][:3]
    for j, idx in enumerate(idxs):
        ax_s = fig.add_axes([0.03 + ci*0.087, 0.58 - j*0.145, 0.075, 0.13])
        ax_s.imshow(X_tr_raw[idx])
        if j == 0:
            ax_s.set_title(cname, color="white", fontsize=7)
        ax_s.axis("off")
        
# Class distribution
ax_dist = fig.add_subplot(gs[0, 2])
_style(fig, ax_dist)
counts = df_eda["class"].value_counts().loc[CLASS_NAMES]
clrs   = sns.color_palette("plasma", 10)
ax_dist.barh(CLASS_NAMES, counts.values, color=clrs, edgecolor="white", lw=0.4)
ax_dist.set_title("Class Distribution", color="white", fontsize=13)
ax_dist.set_xlabel("Count", color="#aaaaaa")
ax_dist.tick_params(colors="white")
for v, bar in zip(counts.values, ax_dist.patches):
    ax_dist.text(bar.get_width()+30, bar.get_y()+bar.get_height()/2,
                 f"{v:,}", va="center", color="white", fontsize=8)


In [ ]:
# Channel histograms
ax_ch = fig.add_subplot(gs[1, :2])
_style(fig, ax_ch)
sample = df_eda.sample(8000, random_state=SEED)
ax_ch.hist(sample["mean_R"], bins=50, color="red",   alpha=0.6, label="Red",   density=True)
ax_ch.hist(sample["mean_G"], bins=50, color="green", alpha=0.6, label="Green", density=True)
ax_ch.hist(sample["mean_B"], bins=50, color="#4488ff",alpha=0.6,label="Blue",  density=True)
ax_ch.set_title("Per-Channel Mean Intensity (8k samples)", color="white", fontsize=13)
ax_ch.set_xlabel("Mean Pixel Value", color="#aaaaaa")
ax_ch.set_ylabel("Density", color="#aaaaaa")
ax_ch.legend(facecolor="#2a2a4a", edgecolor="white", labelcolor="white")


In [ ]:
# Brightness violin
ax_vio = fig.add_subplot(gs[1, 2])
_style(fig, ax_vio)
sns.violinplot(data=df_eda.sample(5000, random_state=SEED),
               x="class", y="brightness", order=CLASS_NAMES,
               palette="plasma", inner="quartile", ax=ax_vio)
ax_vio.set_title("Brightness Distribution by Class", color="white", fontsize=12)
ax_vio.set_xlabel("", color="#aaaaaa")
ax_vio.set_ylabel("Mean Brightness", color="#aaaaaa")
ax_vio.set_xticklabels(CLASS_NAMES, rotation=40, ha="right",
                        color="white", fontsize=8)
ax_vio.tick_params(colors="white")

fig.suptitle("CIFAR-10 — Exploratory Data Analysis",
             color="white", fontsize=18, fontweight="bold", y=0.99)
plt.savefig(f"{OUT}/01_eda.png", dpi=150, bbox_inches="tight",
            facecolor=BG_DARK)
plt.close()
print(f"\n  [✓] EDA plot → {OUT}/01_eda.png")


In [ ]:
# PCA + mean class images

print("  Computing PCA …")
pca_viz  = PCA(n_components=50, random_state=SEED)
flat_n   = flat[:10_000] / 255.0
X_pca_v  = pca_viz.fit_transform(flat_n)

fig, axes = plt.subplots(1, 3, figsize=(21, 6))
_style(fig, *axes)

In [ ]:
# Scree plot
expl = pca_viz.explained_variance_ratio_ * 100
axes[0].bar(range(1, 51), expl, color=ACCENT, alpha=0.8)
axes[0].plot(range(1, 51), np.cumsum(expl), "w-o",
             lw=2, ms=4, label="Cumulative")
axes[0].axhline(80, color=GREEN, ls="--", lw=1.5, label="80%")
axes[0].set_title("PCA Explained Variance", color="white", fontsize=13)
axes[0].set_xlabel("PC", color="#aaaaaa")
axes[0].set_ylabel("Variance %", color="#aaaaaa")
axes[0].legend(facecolor="#2a2a4a", edgecolor="white", labelcolor="white")


In [ ]:
# PC1 vs PC2 scatter
labels_sub = y_tr_raw[:10_000]
for ci in range(10):
    m = labels_sub == ci
    axes[1].scatter(X_pca_v[m, 0], X_pca_v[m, 1],
                    s=5, alpha=0.4, label=CLASS_NAMES[ci])
axes[1].set_title("PCA — PC1 vs PC2", color="white", fontsize=13)
axes[1].set_xlabel("PC1", color="#aaaaaa")
axes[1].set_ylabel("PC2", color="#aaaaaa")
legend = axes[1].legend(markerscale=3, fontsize=7,
                         facecolor="#2a2a4a", edgecolor="white",
                         labelcolor="white", ncol=2)


In [ ]:
# Mean images
axes[2].axis("off")
axes[2].set_title("Mean Image per Class", color="white", fontsize=13)
for ci in range(10):
    mean_img = X_tr_raw[y_tr_raw == ci].mean(0).astype(np.uint8)
    ax_m = fig.add_axes([0.68 + (ci % 5)*0.058,
                          0.08 + (1 - ci//5)*0.38,
                          0.05, 0.3])
    ax_m.imshow(mean_img)
    ax_m.set_title(CLASS_NAMES[ci], color="white", fontsize=7)
    ax_m.axis("off")

fig.suptitle("CIFAR-10 — PCA & Mean Images",
             color="white", fontsize=16, fontweight="bold")
plt.savefig(f"{OUT}/02_pca.png", dpi=150, bbox_inches="tight",
            facecolor=BG_DARK)
plt.close()
print(f"  [✓] PCA plot → {OUT}/02_pca.png")

In [ ]:
# DATA PREPROCESSING
# Normalized float arrays 
X_tr_f  = X_tr_raw.astype("float32") / 255.0   # (50000, 32, 32, 3)
X_te_f  = X_te_raw.astype("float32") / 255.0   # (10000, 32, 32, 3)



In [ ]:
# Flattened (for sklearn)
X_tr_flat = X_tr_f.reshape(50_000, -1)          # (50000, 3072)
X_te_flat = X_te_f.reshape(10_000, -1)          # (10000, 3072)

In [ ]:
# One-hot labels 
y_tr_oh  = keras.utils.to_categorical(y_tr_raw, N_CLASSES)
y_te_oh  = keras.utils.to_categorical(y_te_raw, N_CLASSES)

In [ ]:
# Sklearn subset
SK_TR = 15_000;  SK_TE = 3_000
X_sk_tr = X_tr_flat[:SK_TR];  y_sk_tr = y_tr_raw[:SK_TR]
X_sk_te = X_te_flat[:SK_TE];  y_sk_te = y_te_raw[:SK_TE]

In [ ]:
# Channel-wise normalisation for Keras
ch_mean = X_tr_f.mean(axis=(0,1,2))        # shape (3,)
ch_std  = X_tr_f.std(axis=(0,1,2))         # shape (3,)
X_tr_n  = (X_tr_f - ch_mean) / (ch_std + 1e-7)
X_te_n  = (X_te_f - ch_mean) / (ch_std + 1e-7)
print(f"  Flat shape  : {X_sk_tr.shape}  |  pixel range [{X_sk_tr.min():.2f}, {X_sk_tr.max():.2f}]")
print(f"  Normalised  : mean≈{X_tr_n.mean():.4f}  std≈{X_tr_n.std():.4f}")



In [ ]:
# HELPER UTILITIES
def save_cm(cm, title, fpath, cmap="Blues"):
    fig, ax = plt.subplots(figsize=(11, 9))
    _style(fig, ax)
    cm_pct = cm.astype(float) / cm.sum(1, keepdims=True) * 100
    sns.heatmap(cm_pct, annot=True, fmt=".1f", cmap=cmap,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                linewidths=0.4, linecolor="#2a2a4a",
                cbar_kws={"label":"% True Class"}, ax=ax)
    ax.set_title(title, color="white", fontsize=14, pad=10)
    ax.set_xlabel("Predicted", color="#aaaaaa")
    ax.set_ylabel("True", color="#aaaaaa")
    ax.tick_params(colors="white", labelrotation=30)
    plt.tight_layout()
    plt.savefig(fpath, dpi=130, bbox_inches="tight", facecolor=BG_DARK)
    plt.close()

In [ ]:
def save_history(hist, title, fpath):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    _style(fig, *axes)
    for ax, metric, label in zip(axes, ["loss","accuracy"], ["Loss","Accuracy"]):
        ax.plot(hist.history[metric],        color=ACCENT,  lw=2.5, label="Train")
        ax.plot(hist.history[f"val_{metric}"],color=GREEN, lw=2.5, ls="--", label="Val")
        ax.set_title(f"{title} — {label}", color="white", fontsize=13)
        ax.set_xlabel("Epoch", color="#aaaaaa")
        ax.set_ylabel(label, color="#aaaaaa")
        ax.legend(facecolor="#2a2a4a", edgecolor="white", labelcolor="white")
    plt.tight_layout()
    plt.savefig(fpath, dpi=130, bbox_inches="tight", facecolor=BG_DARK)
    plt.close()

In [ ]:
def run_sklearn(name, model, Xtr, ytr, Xte, yte, pipe=False):
    print(f"\n  ▶ {name}")
    t0 = time.time()
    model.fit(Xtr, ytr)
    elapsed = time.time() - t0
    pred = model.predict(Xte)
    acc  = accuracy_score(yte, pred)
    results[name] = {"accuracy": acc, "train_time": elapsed,
                     "history": None, "type": "sklearn"}
    print(f"    Accuracy : {acc*100:.2f}%   Time : {elapsed:.1f}s")
    return pred, confusion_matrix(yte, pred)



In [ ]:
# BASELINE — LOGISTIC REGRESSION

lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    LogisticRegression(max_iter=1000, C=1.0,
                                   solver="saga", multi_class="multinomial",
                                   random_state=SEED, n_jobs=-1))
])
lr_pred, lr_cm = run_sklearn("Logistic Regression", lr,
                              X_sk_tr, y_sk_tr, X_sk_te, y_sk_te)
save_cm(lr_cm, "Logistic Regression — Confusion Matrix",
        f"{OUT}/03_lr_cm.png", "Blues")

In [ ]:
# Per-class breakdown
lr_report = classification_report(y_sk_te, lr_pred,
                                   target_names=CLASS_NAMES, output_dict=True)
df_lr = pd.DataFrame(lr_report).T.loc[CLASS_NAMES,
                                       ["precision","recall","f1-score"]]
fig, ax = plt.subplots(figsize=(14, 5))
_style(fig, ax)
x, w = np.arange(10), 0.27
ax.bar(x-w,  df_lr["precision"].astype(float), w, label="Precision", color="#4C72B0")
ax.bar(x,    df_lr["recall"].astype(float),    w, label="Recall",    color="#DD8452")
ax.bar(x+w,  df_lr["f1-score"].astype(float),  w, label="F1",        color="#55A868")
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, rotation=30,
                                      ha="right", color="white", fontsize=9)
ax.set_title("Logistic Regression — Per-Class Metrics",
             color="white", fontsize=13)
ax.set_ylim(0, 1.12)
ax.set_ylabel("Score", color="#aaaaaa")
ax.legend(facecolor="#2a2a4a", edgecolor="white", labelcolor="white")
plt.tight_layout()
plt.savefig(f"{OUT}/04_lr_class_metrics.png", dpi=130,
            bbox_inches="tight", facecolor=BG_DARK)
plt.close()
print(f"  [✓] LR plots saved")


In [ ]:
# MODEL 2 — DECISION TREE
dt = DecisionTreeClassifier(max_depth=25, criterion="gini", random_state=SEED)
dt_pred, dt_cm = run_sklearn("Decision Tree (depth=25)", dt,
                              X_sk_tr, y_sk_tr, X_sk_te, y_sk_te)
save_cm(dt_cm, "Decision Tree — Confusion Matrix",
        f"{OUT}/05_dt_cm.png", "Purples")

# Depth vs accuracy
print("  Depth sensitivity …")
depths, daccs = [3,5,8,12,18,25,35], []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=SEED)
    m.fit(X_sk_tr, y_sk_tr)
    daccs.append(accuracy_score(y_sk_te, m.predict(X_sk_te)))
    print(f"    depth={d:3d} → {daccs[-1]*100:.2f}%")
fig, ax = plt.subplots(figsize=(10, 5))
_style(fig, ax)
ax.plot(depths, [a*100 for a in daccs], "o-",
        color="#a29bfe", lw=2.5, ms=9)
ax.fill_between(depths, [a*100 for a in daccs], alpha=0.15, color="#a29bfe")
ax.set_title("Decision Tree — Depth vs Accuracy", color="white", fontsize=13)
ax.set_xlabel("max_depth", color="#aaaaaa")
ax.set_ylabel("Accuracy (%)", color="#aaaaaa")
plt.tight_layout()
plt.savefig(f"{OUT}/06_dt_depth.png", dpi=130,
            bbox_inches="tight", facecolor=BG_DARK)
plt.close()
print(f"  [✓] DT plots saved")



In [ ]:
# MODEL 3 — RANDOM FOREST
rf = RandomForestClassifier(n_estimators=300, max_depth=None,
                             min_samples_split=3, n_jobs=-1, random_state=SEED)
rf_pred, rf_cm = run_sklearn("Random Forest (300 trees)", rf,
                              X_sk_tr, y_sk_tr, X_sk_te, y_sk_te)
save_cm(rf_cm, "Random Forest — Confusion Matrix",
        f"{OUT}/07_rf_cm.png", "Greens")

# Feature importance as image (per channel)
imp = rf.feature_importances_.reshape(32, 32, 3)
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
_style(fig, *axes)
ch_labels = ["Red channel", "Green channel", "Blue channel"]
for ci in range(3):
    im = axes[ci].imshow(imp[:,:,ci], cmap="hot")
    axes[ci].set_title(ch_labels[ci], color="white", fontsize=11)
    axes[ci].axis("off")
    plt.colorbar(im, ax=axes[ci]).ax.yaxis.label.set_color("white")
axes[3].imshow(imp.mean(-1), cmap="inferno")
axes[3].set_title("All-channel mean", color="white", fontsize=11)
axes[3].axis("off")
fig.suptitle("Random Forest — Pixel Importance Maps",
             color="white", fontsize=14)
plt.tight_layout()
plt.savefig(f"{OUT}/08_rf_importance.png", dpi=130,
            bbox_inches="tight", facecolor=BG_DARK)
plt.close()


In [ ]:
# Trees vs accuracy
print("  n_estimators sensitivity …")
n_trees_list, nta = [10, 50, 100, 200, 300], []
for nt in n_trees_list:
    m = RandomForestClassifier(n_estimators=nt, n_jobs=-1, random_state=SEED)
    m.fit(X_sk_tr, y_sk_tr)
    nta.append(accuracy_score(y_sk_te, m.predict(X_sk_te)))
    print(f"    trees={nt:4d} → {nta[-1]*100:.2f}%")
fig, ax = plt.subplots(figsize=(10, 5))
_style(fig, ax)
ax.plot(n_trees_list, [a*100 for a in nta], "o-",
        color=GREEN, lw=2.5, ms=9)
ax.fill_between(n_trees_list, [a*100 for a in nta], alpha=0.15, color=GREEN)
ax.set_title("Random Forest — n_estimators vs Accuracy",
             color="white", fontsize=13)
ax.set_xlabel("Number of Trees", color="#aaaaaa")
ax.set_ylabel("Accuracy (%)", color="#aaaaaa")
plt.tight_layout()
plt.savefig(f"{OUT}/09_rf_ntrees.png", dpi=130,
            bbox_inches="tight", facecolor=BG_DARK)
plt.close()
print(f"  [✓] RF plots saved")

In [ ]:
# MODEL 4 — SVM with HOG-like PCA features
SVM_N = 8_000
scaler_svm = StandardScaler()
pca_svm    = PCA(n_components=200, random_state=SEED)
Xsvm_tr    = pca_svm.fit_transform(scaler_svm.fit_transform(X_sk_tr[:SVM_N]))
Xsvm_te    = pca_svm.transform(scaler_svm.transform(X_sk_te))
y_svm_tr   = y_sk_tr[:SVM_N]

svm = SVC(kernel="rbf", C=10.0, gamma="scale",
          decision_function_shape="ovr", random_state=SEED)
print(f"\n  ▶ SVM (RBF, PCA-200 features, {SVM_N} samples)")
t0 = time.time()
svm.fit(Xsvm_tr, y_svm_tr)
elapsed_svm = time.time() - t0
svm_pred    = svm.predict(Xsvm_te)
svm_acc     = accuracy_score(y_sk_te, svm_pred)
results["SVM (RBF + PCA-200)"] = {"accuracy": svm_acc,
                                    "train_time": elapsed_svm,
                                    "history": None, "type": "sklearn"}
svm_cm = confusion_matrix(y_sk_te, svm_pred)
print(f"    Accuracy : {svm_acc*100:.2f}%   Time : {elapsed_svm:.1f}s")
save_cm(svm_cm, "SVM RBF — Confusion Matrix",
        f"{OUT}/10_svm_cm.png", "YlOrRd")
